<a href="https://colab.research.google.com/github/aidenkoloj/CIRPIN/blob/main/CIRPIN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **CIRPIN: Circular Permutation Invariant Representations For Identifying Putative Protein Homologs**

<img src="https://raw.githubusercontent.com/aidenkoloj/CIRPIN/refs/heads/main/CIRPIN.png" height="300" align="center" style="height:300px">

## What is CIRPIN?

-->  [CIRPIN](https://www.biorxiv.org/content/10.1101/2025.11.18.689110v1) is a search tool for identifying remote homologs of a protein of interest within the [AFDB Cluster Representatives](https://www.nature.com/articles/s41586-023-06510-w#Abs1) and [SCOPe](https://scop.berkeley.edu/) databases. Unlike traditional methods, CIRPIN uses circular permutation-invariant representations to detect structural relationships that other approaches might miss.

## Why use CIRPIN?

-->  Many search methods rely on [dynamic programming](https://en.wikipedia.org/wiki/Smith%E2%80%93Waterman_algorithm) approaches that assume sequential alignment of structures. This limits their ability to detect proteins that may be related by:

- 🔄 **Circular permutation** - where the protein sequence is reordered
- +/- **Insertions and extensions** - additional structural elements
- 🔀 **Domain rewirings** - altered connectivity between structural elements

CIRPIN helps overcome these limitations by learning representations, via our novel data augmentation strategy which generates synCPs, that are invariant to these structural rearrangements.

## What can CIRPIN do for you?

✨ Discover putative homologs that are hidden from conventional sequence and structure search methods due to topological rearrangements

🔍 Search rapidly across millions of protein structures in the AlphaFold Cluster Representatives Database

🧬 Uncover evolutionary relationships and functional similarities between your protein of interest and proteins with similar structure

---

**Citation:**

Kolodziej, A. R., Abulnaga, S. M., & Ovchinnikov, S. (2025). CIRPIN: Learning circular permutation-invariant representations to uncover putative protein homologs. *bioRxiv*. https://doi.org/10.1101/2025.11.18.689110

In [ ]:
#@title Install dependencies
%%time
!pip install -q torch torch_geometric einops
!pip install -q beautifulsoup4
!pip install -q foldcomp
!pip install -q git+https://github.com/sokrypton/py2Dmol.git

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 955.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.3/261.3 kB 5.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 59.8 MB/s eta 0:00:00
CPU times: user 8.7 s, sys: 1.22 s, total: 9.91 s
Wall time: 48.8 s


In [ ]:
#@title Import packages


import os
import torch
from torch.nn import Dropout, Identity, Linear, Sequential, SiLU
from torch.nn.functional import normalize
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from einops import rearrange
from math import ceil
import os
import re
import pickle
import py2Dmol
import requests
from bs4 import BeautifulSoup
import pandas as pd


#@title Download and Compile TM-align
import os

# Download TM-align C++ source code
#print("Downloading TMalign.cpp...")
os.system("wget -qnc https://aideepmed.com/TM-align/TMalign.cpp")

# Compile TM-align
#print("Compiling TMalign...")
# Use -w to suppress warnings, common on Colab for this compilation
os.system("g++ -static -O3 -ffast-math -lm -o TMalign TMalign.cpp -w")

# Check if compilation was successful
#if os.path.exists('./TMalign'):
    #print("TMalign compiled successfully!")
#else:
    #print("Failed to compile TMalign. Please check the output for errors.")


0

In [ ]:
#@markdown ### Download CIRPIN/Progres model weights, embedded databses
print('Downloading CIRPIN/Progres model weights...')
os.system("wget -qnc https://huggingface.co/aidenkzj/CIRPIN/resolve/main/CIRPIN_model_5k_cp_epoch301.pt")
os.system("wget -qnc https://huggingface.co/aidenkzj/CIRPIN/resolve/main/Progres_trained_model.pt")
print('Downloading CIRPIN AFDB_ClustR...')
os.system("wget -qnc https://huggingface.co/datasets/aidenkzj/CIRPIN/resolve/main/combined_embs_3M_CIRPIN.pt")
print('Downloading Progres AFDB_ClustR...')
os.system("wget -qnc https://huggingface.co/datasets/aidenkzj/CIRPIN/resolve/main/combined_embs_3M_progres.pt")
print('Done!')

Done!


In [ ]:
#@title Load Model Weights

#Define model architecture and functions

# Model hyperparameters
n_layers = 6
embedding_size = 128
hidden_dim = 128
hidden_egnn_dim = 64
hidden_edge_dim = 256
pos_embed_dim = 64
n_features = pos_embed_dim + 4
pos_embed_freq_inv = 2000
contact_dist = 10.0  # Å
dropout = 0.0
dropout_final = 0.0


class NoCoordinatesError(Exception):
    pass


class SinusoidalPositionalEncoding(torch.nn.Module):
    def __init__(self, channels):
        super().__init__()
        channels = int(ceil(channels / 2) * 2)
        inv_freq = 1.0 / (pos_embed_freq_inv ** (torch.arange(0, channels, 2).float() / channels))
        self.register_buffer("inv_freq", inv_freq)

    def forward(self, tensor):
        sin_inp_x = torch.einsum("...i,j->...ij", tensor, self.inv_freq)
        emb_x = torch.cat((sin_inp_x.sin(), sin_inp_x.cos()), dim=-1)
        return emb_x


pos_embedder = SinusoidalPositionalEncoding(pos_embed_dim)


def batched_index_select(values, indices, dim=1):
    value_dims = values.shape[(dim + 1):]
    values_shape, indices_shape = map(lambda t: list(t.shape), (values, indices))
    indices = indices[(..., *((None,) * len(value_dims)))]
    indices = indices.expand(*((-1,) * len(indices_shape)), *value_dims)
    value_expand_len = len(indices_shape) - (dim + 1)
    values = values[(*((slice(None),) * dim), *((None,) * value_expand_len), ...)]

    value_expand_shape = [-1] * len(values.shape)
    expand_slice = slice(dim, (dim + value_expand_len))
    value_expand_shape[expand_slice] = indices.shape[expand_slice]
    values = values.expand(*value_expand_shape)

    dim += value_expand_len
    return values.gather(dim, indices)


class EGNN(torch.nn.Module):
    def __init__(self, dim, m_dim=16, dropout=0.0, init_eps=1e-3):
        super().__init__()
        edge_input_dim = (dim * 2) + 1
        dropout = Dropout(dropout) if dropout > 0 else Identity()

        self.edge_mlp = Sequential(
            Linear(edge_input_dim, hidden_edge_dim),
            dropout,
            SiLU(),
            Linear(hidden_edge_dim, m_dim),
            SiLU(),
        )

        self.node_mlp = Sequential(
            Linear(dim + m_dim, dim * 2),
            dropout,
            SiLU(),
            Linear(dim * 2, dim),
        )

        self.init_eps = init_eps
        self.apply(self.init_)

    def init_(self, module):
        if type(module) in {Linear}:
            torch.nn.init.normal_(module.weight, std=self.init_eps)

    def forward(self, feats, coors, mask, adj_mat):
        b, n, d, device = *feats.shape, feats.device

        rel_coors = rearrange(coors, "b i d -> b i () d") - rearrange(coors, "b j d -> b () j d")
        rel_dist = (rel_coors ** 2).sum(dim=-1, keepdim=True)

        i = j = n
        ranking = rel_dist[..., 0].clone()
        rank_mask = mask[:, :, None] * mask[:, None, :]
        ranking.masked_fill_(~rank_mask, 1e5)

        num_nearest = int(adj_mat.float().sum(dim=-1).max().item())
        valid_radius = 0

        self_mask = rearrange(torch.eye(n, device=device, dtype=torch.bool), "i j -> () i j")

        adj_mat = adj_mat.masked_fill(self_mask, False)
        ranking.masked_fill_(self_mask, -1.)
        ranking.masked_fill_(adj_mat, 0.)

        nbhd_ranking, nbhd_indices = ranking.topk(num_nearest, dim=-1, largest=False)
        nbhd_mask = nbhd_ranking <= valid_radius

        rel_coors = batched_index_select(rel_coors, nbhd_indices, dim=2)
        rel_dist = batched_index_select(rel_dist, nbhd_indices, dim=2)

        j = num_nearest
        feats_j = batched_index_select(feats, nbhd_indices, dim=1)
        feats_i = rearrange(feats, "b i d -> b i () d")
        feats_i, feats_j = torch.broadcast_tensors(feats_i, feats_j)

        edge_input = torch.cat((feats_i, feats_j, rel_dist), dim=-1)
        m_ij = self.edge_mlp(edge_input)

        mask_i = rearrange(mask, "b i -> b i ()")
        mask_j = batched_index_select(mask, nbhd_indices, dim=1)
        mask = (mask_i * mask_j) & nbhd_mask

        m_ij_mask = rearrange(mask, "... -> ... ()")
        m_ij = m_ij.masked_fill(~m_ij_mask, 0.)
        m_i = m_ij.sum(dim=-2)

        node_mlp_input = torch.cat((feats, m_i), dim=-1)
        node_out = self.node_mlp(node_mlp_input) + feats

        return node_out, coors


class Model(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.node_enc = Linear(n_features, hidden_dim)
        self.layers = torch.nn.ModuleList()
        for i in range(n_layers):
            self.layers.append(EGNN(
                dim=hidden_dim,
                m_dim=hidden_egnn_dim,
                dropout=dropout,
            ))
        self.node_dec = torch.nn.Sequential(
            Linear(hidden_dim, hidden_dim),
            Dropout(dropout) if dropout > 0 else Identity(),
            SiLU(),
            Linear(hidden_dim, hidden_dim),
        )
        self.graph_dec = torch.nn.Sequential(
            Linear(hidden_dim, hidden_dim),
            Dropout(dropout) if dropout > 0 else Identity(),
            SiLU(),
            Dropout(dropout_final) if dropout_final > 0 else Identity(),
            Linear(hidden_dim, embedding_size),
        )

    def forward(self, data):
        device = data.x.device
        feats, coords = data.x.unsqueeze(0), data.coords.unsqueeze(0)
        adj_mat = torch.sparse_coo_tensor(
            indices=data.edge_index,
            values=torch.tensor([1] * data.edge_index.size(1), device=device),
            size=(data.num_nodes, data.num_nodes),
        ).to_dense().bool().unsqueeze(0)
        mask = torch.ones(1, data.num_nodes, dtype=torch.bool, device=device)

        feats = self.node_enc(feats)
        for layer in self.layers:
            feats, coords = layer(feats, coords, mask, adj_mat)

        feats = self.node_dec(feats)
        # Sum over all nodes to get graph-level features
        graph_feats = feats.squeeze(0).sum(dim=0, keepdim=True)
        out = self.graph_dec(graph_feats)
        return normalize(out, dim=1)


def extract_res_range(rr):
    rr_no_ins_code = re.sub(r"[^0-9-]", "", rr)
    n_hyphen = rr_no_ins_code.count("-")
    if n_hyphen < 3:
        res_start, res_end = rr_no_ins_code.rsplit("-", 1)
    elif n_hyphen == 3:
        splits = rr_no_ins_code.split("-")
        res_start, res_end = "-" + splits[1], "-" + splits[3]
    else:
        raise ValueError(f"could not extract residue range: {rr}")
    return range(int(res_start), int(res_end) + 1)


def get_file_format(fp, fileformat):
    if fileformat == "guess":
        chosen_format = "pdb"
        file_ext = os.path.splitext(fp)[1].lower()
        if file_ext == ".cif" or file_ext == ".mmcif":
            chosen_format = "mmcif"
        elif file_ext == ".mmtf":
            chosen_format = "mmtf"
    else:
        chosen_format = fileformat
    return chosen_format


def read_coords(fp, fileformat="guess", res_range=None):
    chosen_format = get_file_format(fp, fileformat)
    if res_range is None:
        domain_res = None
    else:
        domain_res_list = []
        for rr in res_range.split("_"):
            domain_res_list.extend(extract_res_range(rr))
        domain_res = set(domain_res_list)

    coords = []
    if chosen_format == "pdb":
        with open(fp) as f:
            chain_id = None
            for line in f.readlines():
                if line.startswith("ATOM  ") and line[12:16].strip() == "CA":
                    if chain_id is None:
                        chain_id = line[21]
                    elif line[21] != chain_id:
                        break
                    if domain_res is None or int(line[22:26]) in domain_res:
                        coords.append([float(line[30:38]), float(line[38:46]), float(line[46:54])])
                elif line.startswith("ENDMDL"):
                    break
    elif chosen_format == "mmcif" or chosen_format == "mmtf":
        from Bio.PDB.MMCIFParser import MMCIFParser
        from Bio.PDB.mmtf import MMTFParser

        if chosen_format == "mmcif":
            parser = MMCIFParser()
            struc = parser.get_structure("", fp)
        else:
            struc = MMTFParser.get_structure(fp)

        for model in struc:
            for chain in model:
                for res in chain:
                    if res.get_id()[0] == " ":
                        resnum = res.get_id()[1]
                        for atom in res:
                            if atom.get_name() == "CA":
                                if domain_res is None or resnum in domain_res:
                                    cs = atom.get_coord()
                                    coords.append([float(cs[0]), float(cs[1]), float(cs[2])])
                break
            break
    elif chosen_format == "coords":
        with open(fp) as f:
            c = 0
            for line in f.readlines():
                c += 1
                if domain_res is None or c in domain_res:
                    coords.append([float(v) for v in line.rstrip().split()])
    else:
        raise ValueError("fileformat must be \"guess\", \"pdb\", \"mmcif\", \"mmtf\" or \"coords\"")

    return coords


def coords_to_graph(coords, fp):
    if len(coords) == 0:
        raise NoCoordinatesError(f"No Cα coordinates found in {fp}")

    n_res = len(coords)
    if not isinstance(coords, torch.Tensor):
        coords = torch.tensor(coords)
    if coords.size(1) != 3:
        raise ValueError("coords must be, or must be convertible to, a tensor of shape (nres, 3)")

    dmap = torch.cdist(coords.unsqueeze(0), coords.unsqueeze(0),
                       compute_mode="donot_use_mm_for_euclid_dist")
    contacts = (dmap <= contact_dist).squeeze(0)
    edge_index = contacts.to_sparse().indices()

    degrees = contacts.sum(dim=0)
    norm_degrees = (degrees / degrees.max()).unsqueeze(1)
    term_features = [[0.0, 0.0] for _ in range(n_res)]
    term_features[0][0] = 1.0
    term_features[-1][1] = 1.0
    term_features = torch.tensor(term_features)

    vec_ab = coords[1:-2] - coords[:-3]
    vec_bc = coords[2:-1] - coords[1:-2]
    vec_cd = coords[3:] - coords[2:-1]
    cross_ab_bc = torch.cross(vec_ab, vec_bc, dim=1)
    cross_bc_cd = torch.cross(vec_bc, vec_cd, dim=1)
    taus = torch.atan2(
        (torch.cross(cross_ab_bc, cross_bc_cd, dim=1) * normalize(vec_bc, dim=1)).sum(dim=1),
        (cross_ab_bc * cross_bc_cd).sum(dim=1),
    )
    taus_pad = torch.cat((
        torch.tensor([0.0]),
        taus / torch.pi,
        torch.tensor([0.0, 0.0]),
    )).unsqueeze(1)

    pos_embed = pos_embedder(torch.arange(1, n_res + 1))
    x = torch.cat((norm_degrees, term_features, taus_pad, pos_embed), dim=1)
    data = Data(x=x, edge_index=edge_index, coords=coords)
    return data


def read_graph(fp, fileformat="guess", res_range=None):
    coords = read_coords(fp, fileformat, res_range)
    return coords_to_graph(coords, fp)


def load_trained_model(model_path, device="cpu"):
    """Load the trained model from a checkpoint file."""
    model = Model().to(device)
    loaded_model = torch.load(model_path, map_location=device)
    model.load_state_dict(loaded_model["model"])
    model.eval()
    print(f'Loaded model from {os.path.basename(model_path)}')
    return model


def embed_structure(querystructure, model_path, fileformat="guess", res_range=None, device="cpu"):
    """
    Embed a protein structure using a trained model.

    Args:
        querystructure: Path to structure file (PDB, mmCIF, MMTF, or coords format)
        model_path: Path to the trained model weights (.pt file)
        fileformat: File format ("guess", "pdb", "mmcif", "mmtf", or "coords")
        res_range: Optional residue range string (e.g., "10-50" or "10-50_80-120")
        device: Device to run on ("cpu" or "cuda")

    Returns:
        torch.Tensor: Normalized embedding vector of size (embedding_size,)
    """
    model = load_trained_model(model_path, device)
    graph = read_graph(querystructure, fileformat, res_range)

    with torch.no_grad():
      ### Fix this later; shouldn't need to have a batch because it's just one graph, but may need to adjust some dimensions
        data_loader = DataLoader([graph], batch_size=1)
        for batch in data_loader:
            emb = model(batch.to(device))
            break

    id, notes = get_pdb_info(querystructure)

    embedding_dict = {
            "ids"       : id,
            "embeddings": emb.squeeze(0),
            "nres"      : '',
            "notes"     : notes,
        }

    return embedding_dict

import requests

def get_pdb_info(pdb_file_path):
    """
    Extract PDB ID from a PDB file and lookup its CATH classification.

    Parameters:
    -----------
    pdb_file_path : str
        Path to the PDB file

    Returns:
    --------
    tuple : (id, cath_id)
        id: PDB identifier (4-character code)
        cath_id: CATH classification string (e.g., "3.40.50.360") or "unknown CATH"
    """
    pdb_id = None

    with open(pdb_file_path, 'r') as f:
        for line in f:
            # Extract PDB ID from HEADER line
            if line.startswith('HEADER'):
                # PDB ID is typically in columns 63-66
                if len(line) > 66:
                    pdb_id = line[62:66].strip()

            # Stop reading after ATOM records start (optimization)
            elif line.startswith('ATOM'):
                break

    # If PDB ID not found in HEADER, try to extract from filename
    if not pdb_id:
        import os
        filename = os.path.basename(pdb_file_path)
        # Assume format like "1abc.pdb" or "1ABC.pdb"
        if filename.endswith('.pdb') or filename.endswith('.ent'):
            pdb_id = filename[:4].upper()

    # Lookup CATH classification
    cath_id = lookup_cath_id(pdb_id)

    return pdb_id, cath_id


def lookup_cath_id(pdb_id):
    """
    Lookup CATH classification for a given PDB ID by scraping the CATH website.

    Parameters:
    -----------
    pdb_id : str
        4-character PDB identifier

    Returns:
    --------
    str : CATH classification ID or "unknown CATH"
    """
    if not pdb_id:
        return "unknown CATH"

    try:
        # CATH website URL for the PDB
        url = f"https://www.cathdb.info/pdb/{pdb_id.lower()}"

        response = requests.get(url, timeout=10)

        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'html.parser')

            # Look for the CATH domain table
            # The CATH ID is typically in a table with domain information
            tables = soup.find_all('table')

            for table in tables:
                rows = table.find_all('tr')
                for row in rows:
                    cells = row.find_all(['td', 'th'])
                    for i, cell in enumerate(cells):
                        # Look for cells that contain "CATH" or might be a CATH ID
                        text = cell.get_text(strip=True)
                        # CATH IDs follow pattern like 3.40.50.360
                        if '.' in text and len(text.split('.')) >= 3:
                            parts = text.split('.')
                            if all(part.isdigit() for part in parts):
                                return text

            return "unknown CATH"
        else:
            return "unknown CATH"

    except Exception:
        return "unknown CATH"

def load_embs_labels(pt_fp):
    ''' Load embeddings and labels from saved Progres/CIRPIN pytorch .pt
    Returns:
    embs, labels, caths
    '''
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    data = torch.load(pt_fp, map_location=device, weights_only=False)
    #print(type(data))
    if type(data) != dict:
      embs = data
      labels = 'placeholder'
      caths = 'placeholder'
    else:
      embs = data['embeddings']
      labels = data['ids']

      if all(len(note.split()) > 6 for note in data['notes']):
          caths = [note.split()[6] for note in data['notes']]
      else:
          # Fallback: use first token from each note
          caths = [data['notes']]

      #caths = [note.split()[6] for note in data['notes'][:]]

    print('Loaded embeddings, labels, and caths!', flush=True)

    return embs, labels, caths


prog_model = load_trained_model("Progres_trained_model.pt")
cirpin_model = load_trained_model("CIRPIN_model_5k_cp_epoch301.pt")

# Example usage:
if __name__ == "__main__":
    # Example: Embed a structure
    # embedding = embed_structure(
    #     querystructure="path/to/structure.pdb",
    #     model_path="path/to/trained_model.pt",
    #     device="cpu"
    # )
    # print(f"Embedding shape: {embedding.shape}")
    # print(f"Embedding: {embedding}")
    pass



Loaded model from Progres_trained_model.pt
Loaded model from CIRPIN_model_5k_cp_epoch301.pt


In [ ]:
#@markdown ### Upload or Download PDB
import os
from google.colab import files

#@markdown Choose whether to upload a PDB file or download from RCSB.
upload_or_download = "Download from RCSB" #@param ["Upload PDB", "Download from RCSB"]

filename = None

if upload_or_download == "Upload PDB":
    print("Please upload your PDB file.")
    uploaded = files.upload()
    if uploaded:
        # Get the first (and likely only) uploaded file
        filename = list(uploaded.keys())[0]
        with open(filename, 'wb') as f:
            f.write(uploaded[filename])
        print(f"Successfully uploaded {filename}")
    else:
        print("No file was uploaded.")
elif upload_or_download == "Download from RCSB":
    pdb_id = "9VYS" #@param {type:"string", label:"PDB ID (e.g., 1A0B)"}
    pdb_code = pdb_id.strip().upper()
    if pdb_code:
        url = f"https://files.rcsb.org/download/{pdb_code}.pdb"
        filename = f"{pdb_code}.pdb"
        print(f"Downloading {pdb_code}.pdb from {url}...")
        os.system(f"wget -qnc {url} -O {filename}")
        if os.path.exists(filename):
            print(f"Successfully downloaded {filename}")
        else:
            print(f"Failed to download {filename}. Please check the PDB ID.")
            filename = None # Ensure filename is None if download fails
    else:
        print("Please enter a PDB ID to download.")

if filename:
    print(f"Selected PDB file: {filename}")
else:
    print("No PDB file is available for processing.")



Successfully downloaded 9VYS.pdb
Selected PDB file: 9VYS.pdb


In [ ]:
#@markdown ### Embed query protein
cirpin_embedding = embed_structure(
    querystructure=filename,
    model_path="CIRPIN_model_5k_cp_epoch301.pt",
    device="cpu"
)
torch.save(cirpin_embedding, 'cirpin_query_embedding.pt')
progres_embedding = embed_structure(
    querystructure=filename,
    model_path="Progres_trained_model.pt",
    device="cpu"
)
torch.save(progres_embedding, 'progres_query_embedding.pt')


q_emb, q_l, q_cath = load_embs_labels('progres_query_embedding.pt')


Loaded model from CIRPIN_model_5k_cp_epoch301.pt
Loaded model from Progres_trained_model.pt
Loaded embeddings, labels, and caths!


In [ ]:
#@markdown ### Run Search (~ 1 minute)

# Define file paths for AFDB embedded databases
AFDB_progres_fp = '/content/combined_embs_3M_progres.pt'
AFDB_CIRPIN_fp = '/content/combined_embs_3M_CIRPIN.pt'

#@markdown ### Output file settings
out_list = '/content/putative_pairs.pkl' #@param {type:"string"} # Output file to save putative pairs


def get_putative_pairs_list(progres_AFDB_pt_fp, cirpin_AFDB_pt_fp, query_structure):

    # Load database of embeddings
    progres_emb, progres_labels, progres_caths = load_embs_labels(progres_AFDB_pt_fp)
    cirpin_emb, cirpin_labels, cirpin_caths = load_embs_labels(cirpin_AFDB_pt_fp)

    # Generate query embeddings
    cirpin_embedding = embed_structure(
    querystructure=filename,
    model_path="CIRPIN_model_5k_cp_epoch301.pt",
    device="cpu"
    )
    torch.save(cirpin_embedding, 'cirpin_query_embedding.pt')

    progres_embedding = embed_structure(
        querystructure=filename,
        model_path="Progres_trained_model.pt",
        device="cpu"
    )
    torch.save(progres_embedding, 'progres_query_embedding.pt')


    # Load query embeddings
    q_progres_emb, q_progres_labels, q_p_caths = load_embs_labels('progres_query_embedding.pt')
    q_cirpin_emb, q_cirpin_labels, q_c_caths = load_embs_labels('cirpin_query_embedding.pt')

    # Define empty list of putative CPs
    putative_cps = []

    # Calculate query vs all

    dot_p = q_progres_emb.to(torch.float16) @ progres_emb.T
    scaled_p = (dot_p + 1) / 2


    dot_c = q_cirpin_emb.to(torch.float16) @ cirpin_emb.T
    scaled_c = (dot_c + 1) / 2



    for i in range(len(scaled_p)):
        putative_cps.append([
            q_progres_labels,
            q_p_caths,
            progres_labels[i],
            progres_caths[i],
            float(scaled_p[i].item()),
            float(scaled_c[i].item())
        ])

    print(f'Number of putative CPs: {len(putative_cps)}', flush=True)
    print('Some putative CPs:', flush=True)
    for i, p in enumerate(putative_cps):
        print(p, flush=True)
        if i == 10:
            break

    #print(f'Number of putative pairs: {len(putative_cps)}', flush=True)

    return putative_cps


def save_list(putative_cps, fp):

    with open(fp, 'wb') as f:
        pickle.dump(putative_cps, f)

putative_cps = get_putative_pairs_list(progres_AFDB_pt_fp=AFDB_progres_fp, cirpin_AFDB_pt_fp=AFDB_CIRPIN_fp,
                                       query_structure=filename)

save_list(putative_cps, out_list)

Loaded embeddings, labels, and caths!
Loaded embeddings, labels, and caths!
Loaded model from CIRPIN_model_5k_cp_epoch301.pt
Loaded model from Progres_trained_model.pt
Loaded embeddings, labels, and caths!
Loaded embeddings, labels, and caths!
Number of putative CPs: 3466144
Some putative CPs:
['9VYS', ['unknown CATH'], 'AF-A0A009E921-F1-model_v4_TED01', '1.10.1220', 0.493896484375, 0.31103515625]
['9VYS', ['unknown CATH'], 'AF-A0A009E9H3-F1-model_v4_TED01', '3.40.50', 0.6689453125, 0.30517578125]
['9VYS', ['unknown CATH'], 'AF-A0A009ECR5-F1-model_v4_TED01', '3.60.21.10', 0.40576171875, 0.52392578125]
['9VYS', ['unknown CATH'], 'AF-A0A009ECR5-F1-model_v4_TED02', 'N/A', 0.6533203125, 0.423828125]
['9VYS', ['unknown CATH'], 'AF-A0A009EPC4-F1-model_v4_TED01', '1.10.287', 0.279296875, 0.5869140625]
['9VYS', ['unknown CATH'], 'AF-A0A009EPC4-F1-model_v4_TED02', '3.40.390.10', 0.4140625, 0.52587890625]
['9VYS', ['unknown CATH'], 'AF-A0A009EQS7-F1-model_v4_TED01', '2.40.10', 0.6748046875, 0.58

In [ ]:
#@markdown ### Settings for CIRPIN/Progres score cutoffs
score_diff = 0.05 #@param {type:"number"} # score difference cutoff between Progres/CIRPIN scores
progres_cutoff = 0.6 #@param {type:"number"} # Cutoff for progres scores; must be below this score
cirpin_cutoff = 0.9 #@param {type:"number"} # Cutoff for cirpin scores; must be above this score
cutoff_mode = True #@param {type:"boolean"}

#@markdown Choose a sorting option:
SORT_OPTION = "CIRPIN Score (Desc)" #@param ["None", "Progres Score (Asc)", "Progres Score (Desc)", "CIRPIN Score (Asc)", "CIRPIN Score (Desc)"]
#@markdown OPTIONAL: Enter a CATH ID to filter by (e.g., `2.30.42.10`):
TARGET_CATH_ID = "" #@param {type:"string"}


# Define column names
columns = [
    'Query ID',
    'Query CATH',
    'Target ID',
    'Target CATH',
    'Progres Score',
    'CIRPIN Score'
]

# Create DataFrame from the ORIGINAL putative_cps (don't overwrite it)
df_filtered = pd.DataFrame(putative_cps, columns=columns)

# Convert Query CATH list to string if needed
if df_filtered['Query CATH'].apply(lambda x: isinstance(x, list)).any():
    df_filtered['Query CATH'] = df_filtered['Query CATH'].apply(
        lambda x: x[0] if isinstance(x, list) else x
    )

# Apply filtering based on mode
if cutoff_mode:
    print(f'Using cutoff mode: Progres < {progres_cutoff}, CIRPIN > {cirpin_cutoff}')
    df_filtered = df_filtered[
        (df_filtered['Progres Score'] < progres_cutoff) &
        (df_filtered['CIRPIN Score'] > cirpin_cutoff)
    ]
else:
    print(f'Using score difference mode: (CIRPIN - Progres) > {score_diff}')
    df_filtered = df_filtered[
        (df_filtered['CIRPIN Score'] - df_filtered['Progres Score']) > score_diff
    ]

# Filter by CATH ID if provided
if TARGET_CATH_ID.strip():
    df_filtered = df_filtered[
        df_filtered['Target CATH'] == TARGET_CATH_ID.strip()
    ]
    print(f'Filtered by Target CATH: {TARGET_CATH_ID}')

# Apply sorting
if SORT_OPTION != "None":
    if SORT_OPTION == "Progres Score (Asc)":
        df_filtered = df_filtered.sort_values('Progres Score', ascending=True)
    elif SORT_OPTION == "Progres Score (Desc)":
        df_filtered = df_filtered.sort_values('Progres Score', ascending=False)
    elif SORT_OPTION == "CIRPIN Score (Asc)":
        df_filtered = df_filtered.sort_values('CIRPIN Score', ascending=True)
    elif SORT_OPTION == "CIRPIN Score (Desc)":
        df_filtered = df_filtered.sort_values('CIRPIN Score', ascending=False)
    print(f'Sorted by: {SORT_OPTION}')

print(f'\nTotal entries in putative_cps: {len(putative_cps)}')
print(f'Total putative pairs after filtering: {len(df_filtered)}')

display(df_filtered)

Using cutoff mode: Progres < 0.6, CIRPIN > 0.9
Sorted by: CIRPIN Score (Desc)

Total entries in putative_cps: 3466144
Total putative pairs after filtering: 146


,Query ID,Query CATH,Target ID,Target CATH,Progres Score,CIRPIN Score
112029,9VYS,unknown CATH,AF-A0A0C4DV37-F1-model_v4_TED01,3.30.200,0.564453,0.961914
1933564,9VYS,unknown CATH,AF-A0A5B0MWV5-F1-model_v4_TED01,N/A,0.597656,0.956543
795885,9VYS,unknown CATH,AF-A0A1Y2IE94-F1-model_v4_TED01,N/A,0.571289,0.952148
1934612,9VYS,unknown CATH,AF-A0A5B0QQ73-F1-model_v4_TED01,N/A,0.581055,0.945312
2702891,9VYS,unknown CATH,AF-A0A7X2PNU5-F1-model_v4_TED01,N/A,0.585449,0.944336
...,...,...,...,...,...,...
999388,9VYS,unknown CATH,AF-A0A2H0VGM5-F1-model_v4_TED03,N/A,0.529785,0.900391
1441926,9VYS,unknown CATH,AF-A0A3L6N035-F1-model_v4_TED02,3.30.430,0.563477,0.900391
2189328,9VYS,unknown CATH,AF-A0A6G2PRT8-F1-model_v4_TED01,3.30.70,0.510254,0.900391
3265149,9VYS,unknown CATH,AF-L0IE09-F1-model_v4_TED02,N/A,0.542969,0.900391


In [ ]:
#@title Download AFDB_ClustR using Foldcomp and domain range data  <br> 🚨 Warning: Takes ~ 4 minutes 🕓 setup Foldcomp AFDB-ClustR 🚨
# Takes 4 mins to load AFDB_ClustR

import nest_asyncio
nest_asyncio.apply()

import foldcomp; foldcomp.setup('afdb_rep_v4')


#@title Download cluster reps data
os.system("wget -qnc https://huggingface.co/datasets/aidenkzj/CIRPIN/resolve/main/cluster_reps_ted_365m.domain_summary.cath.globularity.taxid.tsv")

0

In [ ]:
#@title Download top 10 hits and obtain alignment with TM-align/TM-align -cp



#Function to retrieve TED domain


# Load the cluster_reps data once globally for efficiency
import pandas as pd
import subprocess
import os
# Check if cluster_reps_df is already loaded
if 'cluster_reps_df' not in globals():
    print("Loading cluster_reps_df...")
    cluster_reps_df = pd.read_csv('/content/cluster_reps_ted_365m.domain_summary.cath.globularity.taxid.tsv', sep="\t", header=None)
    print("cluster_reps_df loaded.")
else:
    print("cluster_reps_df already loaded.")

def get_domain_data(domains_to_retrieve):
    ''' input: a single TED domain ID (string) '''

    # Filter rows where the first column (index 0) matches the domain to retrieve
    # Use the globally loaded DataFrame
    structure_data = cluster_reps_df[cluster_reps_df[0]==domains_to_retrieve]

    structure_info_dict = {}

    for _, row in structure_data.iterrows():
        cols = row.tolist()
        ted_id = cols[0]
        afdb_id = ted_id.split("_TED")[0]

        # Parse domain residues
        domain_res = set()
        chopping = cols[3]
        domain_res = []

        for res_range in chopping.split("_"):
            res_start, res_end = res_range.split("-")
            domain_res.extend(list(range(int(res_start), int(res_end) + 1)))

        domain_res = set(domain_res)


        structure_info_dict[ted_id] = {
            'afdb_id': afdb_id,
            'chopping': chopping,
            'nres_dom': cols[4],
            'plddt': cols[6],
            'cath_label': cols[13],
            'tax': cols[20],
            'domain_res': domain_res
        }


    return structure_info_dict



def get_pdb(domain_data_dict, ted_id):
    '''
    Get PDB from foldcomp and parse into its domain
    afdb_id: AFDB name without the TED suffix
    domain_res: residues for the input ted_id
    ted_id: AFDB name with TED suffix'''
    #print(f'Length of protein: {len(domain_res)}')


    afdb_id= domain_data_dict[ted_id]['afdb_id']
    domain_res = domain_data_dict[ted_id]['domain_res']
    #print(f'Extracting domain {ted_id}...', flush=True)
    # Get the full protein structure
    temp_afdb_file = f"{afdb_id}.pdb"

    with foldcomp.open("/content/afdb_rep_v4", ids=[afdb_id]) as db:
        for (name, pdb) in db:
            with open(temp_afdb_file, "w") as f:
                f.write(pdb)

    # Extract domain from structure
    print(f'Extracting domain {ted_id}...', flush=True)
    temp_dom_file = f"{ted_id}.pdb"

    with open(temp_afdb_file) as af_struc:
        with open(temp_dom_file, "w") as af_dom:
            for line in af_struc:
                if line.startswith("ATOM"):
                    resnum = int(line[22:26])
                    if resnum in domain_res:
                        af_dom.write(line)
    os.remove(temp_afdb_file)

    return temp_dom_file

def tm_align(query,target, CP=False):
  output_cp_superimposed_pdb = f'{query[:-4]}_{target[:-4]}'

  if os.path.exists('./TMalign') and os.path.exists(query) and os.path.exists(target):
    print(f"Running TM-align on {query} and {target}...")
    try:
        # Run TM-align, telling it to output directly to the specified file

        if CP == False:
          result = subprocess.run(
              ['./TMalign', query, target],
              capture_output=True, text=True, check=True
          )
        elif CP == True:
          result = subprocess.run(
              ['./TMalign', query, target, '-cp', '-o', output_cp_superimposed_pdb],
              capture_output=True, text=True, check=True
          )
        #print(result.stdout)
        if result.stderr:
            print("TM-align encountered warnings or errors:")
            print(result.stderr)

    except subprocess.CalledProcessError as e:
        print(f"TM-align command failed with exit code {e.returncode}.")
        print("STDOUT:")
        print(e.stdout)
        print("STDERR:")
        print(e.stderr)
    #print("TM-align finished. Review the output above for results.")
    print(f"Superimposed structures saved to: {output_cp_superimposed_pdb}")
  else:
      missing_files = []
      if not os.path.exists('./TMalign'):
          missing_files.append('TMalign executable')
      if not os.path.exists(filename):
          missing_files.append(filename)
      if not os.path.exists(q_pdb):
          missing_files.append(q_pdb)
      print(f"Cannot run TM-align. Missing: {', '.join(missing_files)}")

  min_tmscore = get_min_tm_score(result.stdout)

  return min_tmscore




def align_top_hits(df, query_structure, CP=False):
    df_top10 = df.head(10).copy()
    list_of_hits = df_top10["Target ID"].tolist()

    # Add empty column you'll fill
    df_top10["TM-align score"] = None
    df_top10["TM-align CP score"] = None
    df_top10["TM-align score difference"] = None
    for i, hit in enumerate(list_of_hits):
        q_domain_dict = get_domain_data(hit)
        q_pdb = get_pdb(q_domain_dict, hit)
        tmscore = tm_align(query_structure, q_pdb)
        tmscore_CP = tm_align(query_structure, q_pdb, CP=True)
        # Store TM score in the matching row
        df_top10.loc[df_top10.index[i], "TM-align score"] = tmscore
        df_top10.loc[df_top10.index[i], "TM-align CP score"] = tmscore_CP

    df_top10["TM-align score difference"] = df_top10["TM-align CP score"] - df_top10["TM-align score"]
    df_top10_sorted = df_top10.sort_values(by="TM-align score difference", ascending=False)

    return df_top10_sorted

def get_min_tm_score(tm_align_output):

    pdb_chain1, pdb_chain2 = None, None
    tm_score_1, tm_score_2 = None, None
    o = {}

    # Read output line by line
    lines_read = False
    for line in tm_align_output.splitlines():
        lines_read = True  # Check if TM-align produced any output

        if line.startswith("Name of Chain_1:"):
            pdb_chain1 = line.split(':')[1].strip().split()[0]

        elif line.startswith("Name of Chain_2:"):
            pdb_chain2 = line.split(':')[1].strip().split()[0]

        elif "TM-score=" in line:
            if "Chain_1" in line:
                tm_score_1 = float(line.split('=')[1].strip().split()[0])
            elif "Chain_2" in line:
                tm_score_2 = float(line.split('=')[1].strip().split()[0])

    print(f"Chain 1: {pdb_chain1}")
    print(f"Chain 2: {pdb_chain2}")
    print(f"TM-score (Chain 1): {tm_score_1}")
    print(f"TM-score (Chain 2): {tm_score_2}")

    # If no output was read, TM-align likely failed
    if not lines_read:
        raise RuntimeError("Error: TM-align did not produce any output. Check if the TM-align binary is working.")

    # Store results in the dictionary
    if pdb_chain1 and pdb_chain2 and tm_score_1 and tm_score_2:
        o[pdb_chain1] = tm_score_1
        o[pdb_chain2] = tm_score_2

    #print(o)

    # Handle empty dictionary
    if not o:
        raise ValueError("Error: Could not extract TM-scores from output")

    min_tm_score = min(o.values())
    return min_tm_score



top10_hits = align_top_hits(df_filtered,filename)

display(top10_hits)




cluster_reps_df already loaded.
Extracting domain AF-A0A0C4DV37-F1-model_v4_TED01...
Running TM-align on 9VYS.pdb and AF-A0A0C4DV37-F1-model_v4_TED01.pdb...
Superimposed structures saved to: 9VYS_AF-A0A0C4DV37-F1-model_v4_TED01
Chain 1: 9VYS.pdb
Chain 2: AF-A0A0C4DV37-F1-model_v4_TED01.pdb
TM-score (Chain 1): 0.34842
TM-score (Chain 2): 0.38139
Running TM-align on 9VYS.pdb and AF-A0A0C4DV37-F1-model_v4_TED01.pdb...
Superimposed structures saved to: 9VYS_AF-A0A0C4DV37-F1-model_v4_TED01
Chain 1: 9VYS.pdb
Chain 2: AF-A0A0C4DV37-F1-model_v4_TED01.pdb
TM-score (Chain 1): 0.34781
TM-score (Chain 2): 0.37648
Extracting domain AF-A0A5B0MWV5-F1-model_v4_TED01...
Running TM-align on 9VYS.pdb and AF-A0A5B0MWV5-F1-model_v4_TED01.pdb...
Superimposed structures saved to: 9VYS_AF-A0A5B0MWV5-F1-model_v4_TED01
Chain 1: 9VYS.pdb
Chain 2: AF-A0A5B0MWV5-F1-model_v4_TED01.pdb
TM-score (Chain 1): 0.3319
TM-score (Chain 2): 0.33821
Running TM-align on 9VYS.pdb and AF-A0A5B0MWV5-F1-model_v4_TED01.pdb...
Super

,Query ID,Query CATH,Target ID,Target CATH,Progres Score,CIRPIN Score,TM-align score,TM-align CP score,TM-align score difference
1966744,9VYS,unknown CATH,AF-A0A5C7QEX2-F1-model_v4_TED04,N/A,0.557617,0.938965,0.35278,0.43747,0.08469
3425200,9VYS,unknown CATH,AF-W1NXP5-F1-model_v4_TED01,3.30.1360,0.518555,0.941406,0.3886,0.4687,0.0801
2702891,9VYS,unknown CATH,AF-A0A7X2PNU5-F1-model_v4_TED01,N/A,0.585449,0.944336,0.33862,0.4101,0.07148
510474,9VYS,unknown CATH,AF-A0A1F4EP67-F1-model_v4_TED01,N/A,0.580566,0.938965,0.41066,0.46731,0.05665
2260258,9VYS,unknown CATH,AF-A0A6L5XQN9-F1-model_v4_TED02,3.30.1360.60,0.597168,0.940430,0.38496,0.39061,0.00565
1933564,9VYS,unknown CATH,AF-A0A5B0MWV5-F1-model_v4_TED01,N/A,0.597656,0.956543,0.3319,0.3319,0.0
795885,9VYS,unknown CATH,AF-A0A1Y2IE94-F1-model_v4_TED01,N/A,0.571289,0.952148,0.29741,0.29741,0.0
1934612,9VYS,unknown CATH,AF-A0A5B0QQ73-F1-model_v4_TED01,N/A,0.581055,0.945312,0.32661,0.32661,0.0
3136246,9VYS,unknown CATH,AF-E3JS03-F1-model_v4_TED01,N/A,0.564453,0.944336,0.32688,0.32688,0.0
112029,9VYS,unknown CATH,AF-A0A0C4DV37-F1-model_v4_TED01,3.30.200,0.564453,0.961914,0.34842,0.34781,-0.00061


In [ ]:
#@title Retrieve an individual AFDB ClustR TED domain
TED_DOMAIN = "AF-A0A1F4EP67-F1-model_v4_TED01" #@param {type:"string"}

q_domain_dict = get_domain_data(TED_DOMAIN)
q_pdb = get_pdb(q_domain_dict, TED_DOMAIN)


Extracting domain AF-A0A1F4EP67-F1-model_v4_TED01...


## **🔨 PCA -- Work in Progress 🔨**

In [ ]:
# PCA analysis
AFDB_progres_fp = '/content/combined_embs_3M_progres.pt'
AFDB_CIRPIN_fp = '/content/combined_embs_3M_CIRPIN.pt'


def retrieve_domains_for_pca(df, AFDB_cirpin_fp, AFDB_progres_fp):

  # Get list of the df_filtered domains
  domains_to_retrieve = df["Target ID"].tolist()
  progres_emb, progres_labels, progres_caths = load_embs_labels(AFDB_progres_fp)
  cirpin_emb, cirpin_labels, cirpin_caths = load_embs_labels(AFDB_cirpin_fp)


Loaded embeddings, labels, and caths!


In [ ]:
AFDB_progres_fp = '/content/combined_embs_3M_progres.pt'
progres_emb, progres_labels, progres_caths = load_embs_labels(AFDB_progres_fp)
domains_to_retrieve = df_filtered["Target ID"].tolist()
indices_dict = {v: i for i, v in enumerate(progres_labels)}

# Get indices for your values
indices = [indices_dict[v] for v in domains_to_retrieve if v in indices_dict]

Loaded embeddings, labels, and caths!


In [ ]:
type(progres_labels)

list

In [ ]:
df_filtered

,Query ID,Query CATH,Target ID,Target CATH,Progres Score,CIRPIN Score
112029,9VYS,unknown CATH,AF-A0A0C4DV37-F1-model_v4_TED01,3.30.200,0.564453,0.961914
1933564,9VYS,unknown CATH,AF-A0A5B0MWV5-F1-model_v4_TED01,N/A,0.597656,0.956543
795885,9VYS,unknown CATH,AF-A0A1Y2IE94-F1-model_v4_TED01,N/A,0.571289,0.952148
1934612,9VYS,unknown CATH,AF-A0A5B0QQ73-F1-model_v4_TED01,N/A,0.581055,0.945312
2702891,9VYS,unknown CATH,AF-A0A7X2PNU5-F1-model_v4_TED01,N/A,0.585449,0.944336
...,...,...,...,...,...,...
999388,9VYS,unknown CATH,AF-A0A2H0VGM5-F1-model_v4_TED03,N/A,0.529785,0.900391
1441926,9VYS,unknown CATH,AF-A0A3L6N035-F1-model_v4_TED02,3.30.430,0.563477,0.900391
2189328,9VYS,unknown CATH,AF-A0A6G2PRT8-F1-model_v4_TED01,3.30.70,0.510254,0.900391
3265149,9VYS,unknown CATH,AF-L0IE09-F1-model_v4_TED02,N/A,0.542969,0.900391


In [ ]:
q_progres_labels

'9VYS'

In [ ]:
df_filtered

,Query ID,Query CATH,Target ID,Target CATH,Progres Score,CIRPIN Score
112029,9VYS,unknown CATH,AF-A0A0C4DV37-F1-model_v4_TED01,3.30.200,0.564453,0.961914
1933564,9VYS,unknown CATH,AF-A0A5B0MWV5-F1-model_v4_TED01,N/A,0.597656,0.956543
795885,9VYS,unknown CATH,AF-A0A1Y2IE94-F1-model_v4_TED01,N/A,0.571289,0.952148
1934612,9VYS,unknown CATH,AF-A0A5B0QQ73-F1-model_v4_TED01,N/A,0.581055,0.945312
2702891,9VYS,unknown CATH,AF-A0A7X2PNU5-F1-model_v4_TED01,N/A,0.585449,0.944336
...,...,...,...,...,...,...
999388,9VYS,unknown CATH,AF-A0A2H0VGM5-F1-model_v4_TED03,N/A,0.529785,0.900391
1441926,9VYS,unknown CATH,AF-A0A3L6N035-F1-model_v4_TED02,3.30.430,0.563477,0.900391
2189328,9VYS,unknown CATH,AF-A0A6G2PRT8-F1-model_v4_TED01,3.30.70,0.510254,0.900391
3265149,9VYS,unknown CATH,AF-L0IE09-F1-model_v4_TED02,N/A,0.542969,0.900391


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title Visualize superimposed structures using py2Dmol


import py2Dmol

# 1. Create a viewer object
#viewer1 = py2Dmol.view(size=(600, 600))
#viewer2 = py2Dmol.view(size=(600, 600))
viewer3 = py2Dmol.view(size=(600, 600))
# 2. Add pdb
#viewer1.add_pdb(q_pdb, chains=['A'])
#viewer2.add_pdb(filename, chains=['A'])
viewer3.add_pdb('/content/superimposed_structures_for_visualization.pdb', chains=['A'])
viewer3.add_pdb('/content/AF-A0A7S3NQ21-F1-model_v4_TED02.pdb', chains=['A'])
# 3. Show the final, static viewer
#viewer1.show()
#viewer2.show()
viewer3.show()



Error reading structure /content/AF-A0A7S3NQ21-F1-model_v4_TED02.pdb: [Errno 2] Failed to open /content/AF-A0A7S3NQ21-F1-model_v4_TED02.pdb: No such file or directory
